Running IsoVita pipeline on DLPFC data - figures are generated in another notebook

**Data loading and preprocessing**

In [6]:
from pathlib import Path

import pandas as pd

from prepare_isovita_inputs import (
    prepare_isovita_inputs_from_expression,
    save_prepared_inputs,
)


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path("/users/xchen5/IsoVita/data/DLPFC")

EXPRESSION_PATH = DATA_DIR / "tximport_counts.txt"
METADATA_PATH = DATA_DIR / "metadata.tsv"
TX2GENE_PATH = DATA_DIR / "tx2gene.csv"
OUTPUT_DIR = DATA_DIR / "isovita_inputs"


# ============================================================
# Load inputs
# ============================================================

expression = pd.read_csv(
    EXPRESSION_PATH,
    sep=r"\s+",
)

metadata = pd.read_csv(
    METADATA_PATH,
    sep="\t",
)

tx2gene = pd.read_csv(
    TX2GENE_PATH,
)


# ============================================================
# Basic expression checks
# ============================================================

assert expression.columns[0] == "tx_id"
assert expression["tx_id"].is_unique

print("Expression shape:", expression.shape)
print("First sample columns:", expression.columns[1:6].tolist())
print(expression.iloc[:5, :6])


# ============================================================
# Harmonize DLPFC metadata sample IDs
# ============================================================

metadata = metadata.copy()

metadata["dlpfc_sample_id"] = (
    metadata["sample_id"]
    .astype(str)
    .str.split("_", n=1)
    .str[0]
)


# ============================================================
# Restrict metadata to DLPFC
# ============================================================

dlpfc_metadata = metadata.loc[
    metadata["region"].astype(str).eq("DLPFC")
].copy()


# ============================================================
# Validate duplicate sequencing/library rows
# ============================================================

duplicate_mask = (
    dlpfc_metadata["dlpfc_sample_id"]
    .duplicated(keep=False)
)

duplicate_metadata = (
    dlpfc_metadata.loc[duplicate_mask]
    .copy()
)

if not duplicate_metadata.empty:

    age_counts = (
        duplicate_metadata
        .groupby("dlpfc_sample_id")["age"]
        .nunique(dropna=False)
    )

    conflicting_age_ids = (
        age_counts.loc[age_counts > 1]
        .index
        .tolist()
    )

    if conflicting_age_ids:
        raise ValueError(
            "Some duplicated DLPFC sample IDs have conflicting ages. "
            f"Examples: {conflicting_age_ids[:10]}"
        )

    print(
        f"\nDuplicated sequencing rows detected for "
        f"{duplicate_metadata['dlpfc_sample_id'].nunique()} "
        "DLPFC biological samples."
    )

    print(
        "All duplicate rows agree on age."
    )


# ============================================================
# Collapse to one metadata row per biological sample
# ============================================================

dlpfc_metadata = (
    dlpfc_metadata
    .drop_duplicates(
        subset="dlpfc_sample_id",
        keep="first",
    )
    .reset_index(drop=True)
)


# ============================================================
# Check overlap with expression matrix
# ============================================================

expr_sample_ids = set(
    expression.columns[1:].astype(str)
)

meta_sample_ids = set(
    dlpfc_metadata["dlpfc_sample_id"].astype(str)
)

shared_sample_ids = (
    expr_sample_ids
    & meta_sample_ids
)

print("\nExpression samples:", len(expr_sample_ids))
print("Unique DLPFC metadata samples:", len(meta_sample_ids))
print("Shared samples:", len(shared_sample_ids))
print(
    "Expression samples without metadata:",
    len(expr_sample_ids - meta_sample_ids),
)
print(
    "Metadata samples without expression:",
    len(meta_sample_ids - expr_sample_ids),
)

if not shared_sample_ids:
    raise ValueError(
        "No overlapping DLPFC sample IDs after metadata harmonization."
    )


# ============================================================
# Prepare canonical IsoVita inputs
# ============================================================

prepared = prepare_isovita_inputs_from_expression(
    expression=expression,
    metadata=dlpfc_metadata,
    tx2gene=tx2gene,
    metadata_sample_col="dlpfc_sample_id",
    age_col="age",
    metadata_filters=None,
    expression_transcript_col="tx_id",
    tx2gene_transcript_col="transcript_id",
    tx2gene_gene_col="gene_id",
    postnatal_only=True,
)


# ============================================================
# Final alignment check
# ============================================================

tpm_sample_cols = [
    c
    for c in prepared.tpm.columns
    if c not in {
        "gene_id",
        "transcript_id",
    }
]

assert (
    tpm_sample_cols
    == prepared.metadata["sample_id"]
    .astype(str)
    .tolist()
)


# ============================================================
# Save canonical inputs
# ============================================================

save_prepared_inputs(
    prepared,
    OUTPUT_DIR,
)


# ============================================================
# Preparation summary
# ============================================================

print("\nPreparation summary:")
print(prepared.summary.to_string(index=False))

print(f"\nSaved to: {OUTPUT_DIR}")

Expression shape: (157350, 342)
First sample columns: ['R2824', 'R2826', 'R2835', 'R2836', 'R2839']
          tx_id       R2824        R2826        R2835       R2836        R2839
0       CHS.1.1    3.083703     0.000000     0.000000    0.000000     0.000000
1       CHS.2.1  658.034576  1094.908037  1521.988337  985.056726  1307.761983
2  CHS.166734.2    0.000000     0.000000     0.000000    0.000000     0.000000
3  CHS.166734.1    0.000000     0.000000     0.000000    0.000000     0.000000
4  CHS.166734.3    0.000000     0.000000     0.000000    0.000000     0.000000

Duplicated sequencing rows detected for 210 DLPFC biological samples.
All duplicate rows agree on age.

Expression samples: 341
Unique DLPFC metadata samples: 1036
Shared samples: 341
Expression samples without metadata: 0
Metadata samples without expression: 695

Preparation summary:
              metric     value
    retained_samples    285.00
retained_transcripts 157310.00
      retained_genes  57921.00
             ag

**Running `IsoVita` pipeline**

In [7]:
# ============================================================
# Running IsoVita pipeline
# ============================================================

from pathlib import Path
import shutil

import pandas as pd

from isovita import (
    IsoVitaConfig,
    PermutationInferenceConfig,
    run_isovita_pipeline,
    save_isovita_pipeline_outputs,
)


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path(
    "/users/xchen5/IsoVita/data/DLPFC"
)

INPUT_DIR = (
    DATA_DIR
    / "isovita_inputs"
)

OUTPUT_DIR = (
    DATA_DIR
    / "isovita_results"
)


# ============================================================
# Load prepared inputs
# ============================================================

tpm = pd.read_csv(
    INPUT_DIR
    / "tpm.tsv",
    sep="\t",
)

metadata = pd.read_csv(
    INPUT_DIR
    / "metadata.tsv",
    sep="\t",
)


# ============================================================
# IsoVita configuration
# ============================================================

config = IsoVitaConfig(
    inference=PermutationInferenceConfig(
        n_permutations=5000,
        random_seed=0,
        batch_size=128,
    ),
)


# ============================================================
# Run production IsoVita
# ============================================================

result = run_isovita_pipeline(
    tpm=tpm,
    metadata=metadata,
    config=config,
    progress_every=1000,  # progress report
    n_jobs=16,  # shared-memory worker threads for permutation inference
)


# ============================================================
# Save outputs
# ============================================================

if OUTPUT_DIR.exists():
    shutil.rmtree(
        OUTPUT_DIR
    )


save_isovita_pipeline_outputs(
    result,
    OUTPUT_DIR,
)


# ============================================================
# Run summary
# ============================================================

print(
    "\nIsoVita run complete"
)

print(
    f"Output directory: "
    f"{OUTPUT_DIR}"
)


print(
    f"\nGenes entering balance construction: "
    f"{(
        result.gene_routes[
            'route'
        ]
        == 'balance_ge2'
    ).sum():,}"
)


print(
    f"Successful balance features: "
    f"{(
        result.balance_summary[
            'status'
        ]
        == 'ok'
    ).sum():,}"
)


print(
    f"Discrete discoveries: "
    f"{len(
        result.discrete_trajectory_candidates
    ):,}"
)


print(
    f"Continuous discoveries: "
    f"{len(
        result.continuous_trajectory_candidates
    ):,}"
)


print(
    f"Unique genes in production-branch union: "
    f"{len(
        result.final_features
    ):,}"
)

Balance: 0/11404
Balance: 1000/11404
Balance: 2000/11404
Balance: 3000/11404
Balance: 4000/11404
Balance: 5000/11404
Balance: 6000/11404
Balance: 7000/11404
Balance: 8000/11404
Balance: 9000/11404
Balance: 10000/11404
Balance: 11000/11404
Selection-aware inference: 0/11404
Selection-aware inference: 1000/11404
Selection-aware inference: 2000/11404
Selection-aware inference: 3000/11404
Selection-aware inference: 4000/11404
Selection-aware inference: 5000/11404
Selection-aware inference: 6000/11404
Selection-aware inference: 7000/11404
Selection-aware inference: 8000/11404
Selection-aware inference: 9000/11404
Selection-aware inference: 10000/11404
Selection-aware inference: 11000/11404


/users/xchen5/IsoVita/isovita.py:2787: RuntimeWarning: Discrete permutation resolution may be coarse for BH: minimum p=0.0002, first BH threshold=4.38e-06. Increase inference.n_permutations when precise tail resolution is required.
  out["discrete"] = _finalize_discrete_table(
/users/xchen5/IsoVita/isovita.py:2794: RuntimeWarning: Continuous permutation resolution may be coarse for BH: minimum p=0.0002, first BH threshold=4.38e-06. Increase inference.n_permutations when precise tail resolution is required.
  out["continuous"] = _finalize_continuous_table(



IsoVita run complete
Output directory: /users/xchen5/IsoVita/data/DLPFC/isovita_results

Genes entering balance construction: 11,404
Successful balance features: 11,404
Discrete discoveries: 1,618
Continuous discoveries: 2,562
Unique genes in production-branch union: 2,765
